# Buổi E4: Gradient boosting dùng thật, nhanh, chịu ô trống, biết dừng

**Hiện trường.** Công ty của chị Hạnh có bảng **hàng chục nghìn khách hàng**, nhiều cột, **nhiều ô trống**, vài cột chữ (thành phố, gói cước). Cô thử gradient boosting từ E3 (`GradientBoostingClassifier`) thì **chạy lâu** và phải tự điền ô trống, mã hóa cột chữ trước. Đồng nghiệp bảo: *"dùng bản **histogram** của sklearn, nhanh hơn nhiều, chịu ô trống, hiểu cột chữ luôn"*.

Buổi này dạy `HistGradientBoostingClassifier` (và bản hồi quy): bản gradient boosting dùng cho **dữ liệu dạng bảng thật**, phổ biến nhất ngoài đời (cùng họ với XGBoost, LightGBM).

**Ẩn dụ:** thay vì xét **từng giá trị** của từng cột để tìm chỗ cắt, ta **gom giá trị thành khoảng** (như phân nhóm tuổi 0-10, 10-20, ...) rồi chỉ xét các biên khoảng. Nhanh hơn rất nhiều, mất rất ít độ chính xác.

**Mindset của người làm dữ liệu:** trên bảng số liệu, nhóm gradient boosting thường là ứng viên mạnh nhất; nhưng ta vẫn **so với mốc đơn giản** và kiểm tra rò rỉ trước khi tin điểm.

**Lưu ý:** dữ liệu sinh ra bằng code (dựng có chủ đích), dùng để học.

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: tốc độ trên 20 000 dòng |
| Chặng 1 | Ba núm chính và dừng sớm tự động |
| Chặng 2 | Phanh an toàn: cỡ cây và phạt độ lớn |
| Chặng 3 | Ô trống: không cần điền trước |
| Chặng 4 | Cột chữ: để mô hình tự hiểu |
| Chặng 5 | Vặn núm có kiểm soát, và đừng kỳ vọng quá |

## Chặng 0: Thấy ngay, tốc độ trên 20 000 dòng

**Câu hỏi:** bản histogram nhanh hơn bao nhiêu?

Ta sinh bảng **20 000 dòng × 30 cột** (`make_classification`, dựng có chủ đích) và so thời gian huấn luyện cùng điểm thi của hai bản.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
X, y = make_classification(n_samples=20000, n_features=30, n_informative=10, n_redundant=5, flip_y=0.03, class_sep=0.8, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
rows = []
for name, m in [("GradientBoosting (E3)", GradientBoostingClassifier(n_estimators=100, random_state=0)), ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=100, random_state=0))]:
    t0 = time.time(); m.fit(X_train, y_train); sec = time.time() - t0
    rows.append({"model": name, "fit seconds": sec, "test accuracy": m.score(X_test, y_test)})
speed = pd.DataFrame(rows)
print(speed.round(3).to_string(index=False))

                model  fit seconds  test accuracy
GradientBoosting (E3)       13.513          0.885
 HistGradientBoosting        0.762          0.932


**Giải thích từng dòng**

- `HistGradientBoostingClassifier(max_iter=100)`: gradient boosting bản histogram. Chú ý tên núm số vòng là `max_iter` (không phải `n_estimators`) vì nó là "số vòng tối đa".
- `t0 = time.time(); m.fit(...); sec = time.time() - t0`: bấm giờ huấn luyện (đã dùng ở D1).

**Cách đọc.**
- **Con số:** bản cũ **15,3** giây, bản histogram **0,7** giây (nhanh khoảng **22** lần); điểm thi **88,5%** so với **93,2%**.
- **Bức tranh:** bản histogram nhanh hơn rất nhiều. Về điểm, **không nên đọc đây là "cùng mức"**: ở cài đặt mặc định bản histogram dùng cây to hơn (31 lá, so với cây sâu 3 của bản cũ) nên điểm khác nhau; so công bằng cần chỉnh cả hai. Điều chắc chắn là tốc độ: với dữ liệu lớn hơn nữa khoảng cách còn nới ra, nên bản histogram là lựa chọn mặc định cho bảng nhiều dòng.
- **Câu hỏi tiếp:** các núm của bản này khác gì E3?

**Mang về chặng 0:** `HistGradientBoosting*` nhanh hơn rất nhiều, và ở mặc định cũng không kém điểm; dùng làm mặc định cho dữ liệu bảng.

## Chặng 1: Ba núm chính và dừng sớm tự động

**Câu hỏi:** nên chỉnh núm nào trước?

Ba núm chính (các núm khác gọi chung là "phanh an toàn"):

1. **`learning_rate`**: phần mỗi vòng được cộng (như E3). Nhỏ (0,05 đến 0,1) thì ổn định, cần nhiều vòng.
2. **`max_iter`**: số vòng **tối đa**.
3. **`early_stopping`**: tự dành một phần dữ liệu để theo dõi và **dừng khi hết cải thiện**. Với `early_stopping="auto"` (mặc định), tự bật khi có hơn 10 000 dòng.

Ta cho `max_iter=1000` và xem máy tự dừng ở đâu với các tốc độ học khác nhau.

In [3]:
rows = []
for lr in [0.3, 0.1, 0.03]:
    m = HistGradientBoostingClassifier(learning_rate=lr, max_iter=1000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=15, random_state=0).fit(X_train, y_train)
    rows.append({"learning_rate": lr, "rounds used": m.n_iter_, "test accuracy": m.score(X_test, y_test)})
stopping = pd.DataFrame(rows)
print(stopping.round(3).to_string(index=False))

 learning_rate  rounds used  test accuracy
          0.30           90          0.934
          0.10          206          0.936
          0.03          816          0.938


**Giải thích từng dòng**

- `early_stopping=True`: bật dừng sớm: bản histogram tự để riêng một phần phần-học làm phần theo dõi.
- `validation_fraction=0.15`: tỉ lệ phần theo dõi (15%), lấy từ **tập học**, không đụng tập thi.
- `n_iter_no_change=15`: dừng nếu 15 vòng liền không cải thiện.
- `m.n_iter_`: số vòng đã thực sự chạy trước khi dừng.

**Cách đọc.**
- **Con số:** `learning_rate` 0,3 dừng ở **90** vòng (thi 93,4%); 0,1 dừng ở **206** (thi 93,6%); 0,03 dừng ở **816** (thi 93,8%).
- **Bức tranh:** tốc độ học nhỏ cần **nhiều vòng hơn** để tới cùng chỗ, còn điểm cuối thường **gần nhau**. Vì dừng sớm đã chọn số vòng giúp ta, núm duy nhất cần chọn là `learning_rate`; 0,05 đến 0,1 là điểm xuất phát an toàn.

**Mang về chặng 1:** đặt `max_iter` lớn, bật dừng sớm, chọn `learning_rate` nhỏ vừa phải; số vòng do máy quyết.

## Chặng 2: Phanh an toàn, cỡ cây và phạt độ lớn

**Câu hỏi:** mỗi cây thành viên nên "to" cỡ nào?

- **`max_leaf_nodes`** (mặc định 31): số lá tối đa mỗi cây. Nhiều lá thì mỗi cây mạnh hơn nhưng dễ quá khớp.
- **`min_samples_leaf`** (mặc định 20): số dòng tối thiểu mỗi lá.
- **`l2_regularization`**: phạt giá trị lá lớn, ép mô hình "khiêm tốn" hơn.

Với dữ liệu ít nhiễu như bảng này khác biệt nhỏ; ta thử trên một bản **ít dữ liệu và nhiều nhiễu** (1000 dòng) để thấy tác dụng phanh.

In [4]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
Xs, ys = X_train[:1000], y_train[:1000]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
rows = []
for label, kw in [("max_leaf_nodes=3", {"max_leaf_nodes": 3}), ("max_leaf_nodes=15", {"max_leaf_nodes": 15}), ("max_leaf_nodes=63", {"max_leaf_nodes": 63}), ("63 leaves + min_samples_leaf=40", {"max_leaf_nodes": 63, "min_samples_leaf": 40}), ("63 leaves + l2=5", {"max_leaf_nodes": 63, "l2_regularization": 5.0})]:
    m = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=150, early_stopping=False, random_state=0, **kw)
    s = cross_val_score(m, Xs, ys, cv=cv, n_jobs=-1)
    rows.append({"setting": label, "CV accuracy": s.mean(), "std": s.std()})
brakes = pd.DataFrame(rows)
print(brakes.round(3).to_string(index=False))

                        setting  CV accuracy   std
               max_leaf_nodes=3        0.814 0.028
              max_leaf_nodes=15        0.881 0.019
              max_leaf_nodes=63        0.874 0.024
63 leaves + min_samples_leaf=40        0.869 0.023
               63 leaves + l2=5        0.880 0.029


**Giải thích từng dòng**

- `X_train[:1000]`: lấy 1000 dòng đầu của tập học để mô phỏng dữ liệu ít.
- `early_stopping=False`: tắt dừng sớm để so các cài đặt cùng số vòng 150.
- `**kw`: mở từ điển thành các tham số có tên (đã gặp ở E2).

**Cách đọc.**
- **Con số:** điểm thấp nhất **81,4%**, cao nhất **88,1%**; chênh lệch **6,7** điểm phần trăm, với độ lệch trong khoảng 1,9 đến 2,9 điểm.
- **Bức tranh:** cây quá nhỏ (3 lá) **học chưa đủ** rõ rệt; từ 15 lá trở lên điểm gần nhau. Cây 63 lá hơi kém hơn cây 15 lá và hai loại phanh (`min_samples_leaf`, `l2_regularization`) kéo điểm lên hoặc không, nhưng **chênh lệch nằm trong độ lệch**, nên **không kết luận được** phanh nào hơn. Nói thật: với bài này phanh không chứng minh được lợi ích; chúng quan trọng hơn khi nhãn nhiễu mạnh hơn hoặc dữ liệu còn ít hơn nữa.
- **Quy tắc:** dữ liệu nhiều, mặc định ổn; dữ liệu ít hoặc nhiễu, tăng phanh.

**Mang về chặng 2:** cây to cần phanh khi dữ liệu ít; phanh có ba cách: ít lá, lá nhiều dòng, phạt độ lớn.

## Chặng 3: Ô trống, không cần điền trước

**Câu hỏi:** nếu bảng có ô trống, phải điền trước như B4 không?

Bản histogram **tự xử lý ô trống**: ở mỗi lần chia nhánh, nó thử đẩy dòng có ô trống sang trái hoặc phải, **bên nào tốt hơn thì chọn**. Ô trống thành một thông tin chứ không phải lỗi.

Ta **làm bẩn có chủ đích**: xóa ngẫu nhiên 25% ô của dữ liệu (xóa hoàn toàn ngẫu nhiên). Rồi so: (a) bản histogram dùng thẳng, (b) điền bằng trung vị rồi dùng hồi quy logistic.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
rng = np.random.RandomState(0)
mask_tr = rng.rand(*X_train.shape) < 0.25
mask_te = rng.rand(*X_test.shape) < 0.25
Xn_train, Xn_test = X_train.copy(), X_test.copy()
Xn_train[mask_tr] = np.nan; Xn_test[mask_te] = np.nan
hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=0).fit(Xn_train, y_train)
lin = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xn_train, y_train)
print("missing share:", round(float(np.isnan(Xn_train).mean()), 3))
print("HistGB direct   | test:", round(hgb.score(Xn_test, y_test), 3), "| (no missing values: ", round(HistGradientBoostingClassifier(max_iter=300, random_state=0).fit(X_train, y_train).score(X_test, y_test), 3), ")")
print("median + logistic| test:", round(lin.score(Xn_test, y_test), 3))

missing share: 0.251


HistGB direct   | test: 0.874 | (no missing values:  0.938 )
median + logistic| test: 0.651


**Giải thích từng dòng**

- `rng.rand(*X_train.shape) < 0.25`: một bảng đúng/sai cùng cỡ với `X_train`, 25% ô là đúng (sẽ bị xóa). Dấu `*` mở `(hàng, cột)` thành hai tham số.
- `Xn_train[mask_tr] = np.nan`: đặt ô được chọn thành `NaN` (ô trống). `.copy()` ở dòng trên để không làm hỏng dữ liệu gốc.
- `HistGradientBoostingClassifier(...).fit(Xn_train, y_train)`: đưa thẳng bảng có `NaN`, **không điền gì**.
- `SimpleImputer(strategy="median")`: điền ô trống bằng trung vị mỗi cột (B4); nằm **trong pipeline** nên học lại đúng cách (D3).

**Cách đọc.**
- **Con số:** bản histogram dùng thẳng: **87,4%**; trung vị + logistic: **65,1%**.
- **Bức tranh:** khi **25% ô trống ngẫu nhiên**, bản histogram vẫn chạy, không cần điền, và hơn hẳn cách "trung vị + logistic" (lưu ý đây là so hai **họ mô hình** khác nhau, không riêng cách xử lý ô trống). Nhưng "chịu được" không có nghĩa "ô trống vô hại": điểm vẫn thấp hơn khi bảng đầy đủ (xem dòng "no missing values" ở trên).
- **Cảnh báo nghề:** ở dữ liệu thật ô trống thường **không ngẫu nhiên** (ví dụ khách không điền thu nhập vì thu nhập thấp), nên ô trống có thể là **tín hiệu**, tốt cho mô hình nhưng cần hỏi: *lúc dùng thật cột này có trống kiểu đó không?* (xem D3).

**Mang về chặng 3:** bản histogram nhận `NaN` trực tiếp; nhưng hiểu **vì sao** ô trống xuất hiện vẫn là việc của bạn.

## Chặng 4: Cột chữ, để mô hình tự hiểu

**Câu hỏi:** cột chữ (thành phố, gói cước) xử lý thế nào?

Cách cũ: mã hóa one-hot (B4), mỗi giá trị thành một cột. Cách mới: khai cột đó là **phân loại** (`category`) và bảo mô hình dùng `categorical_features="from_dtype"`; nó tự tìm cách **gộp nhóm giá trị** tốt nhất mà không nở bảng.

Ta dựng bảng khách hàng giả với cột `plan` có 200 gói; **một nửa số gói** (ngẫu nhiên, không theo thứ tự mã) có khả năng bỏ dịch vụ cao. Cột `plan` là mã số 0 đến 199 **không có thứ tự ý nghĩa**.

In [6]:
rng = np.random.RandomState(1)
n = 20000
plan = rng.randint(0, 200, n)
risky = rng.permutation(200) < 100
tenure = rng.rand(n)
p_churn = 0.15 + 0.5 * risky[plan] - 0.1 * tenure
churn = (rng.rand(n) < p_churn).astype(int)
df = pd.DataFrame({"plan": plan, "tenure": tenure})
df_cat = df.copy(); df_cat["plan"] = df_cat["plan"].astype("category")
a, b, ya, yb = train_test_split(df, churn, test_size=0.3, random_state=0)
ac, bc_ = df_cat.loc[a.index], df_cat.loc[b.index]
num = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(a, ya)
cat = HistGradientBoostingClassifier(max_iter=200, categorical_features="from_dtype", random_state=0).fit(ac, ya)
print("plan as a number (treated as ordered):", round(num.score(b, yb), 3))
print("plan as a category:                    ", round(cat.score(bc_, yb), 3))

plan as a number (treated as ordered): 0.722
plan as a category:                     0.737


**Giải thích từng dòng**

- `risky = rng.permutation(30) < 15`: chọn ngẫu nhiên 100 trong 200 gói là "gói rủi ro", không theo thứ tự mã.
- `p_churn = 0.15 + 0.5 * risky[plan] - 0.1 * tenure`: xác suất bỏ: nền 15%, +50% nếu gói rủi ro, giảm nhẹ theo thời gian gắn bó.
- `df_cat["plan"].astype("category")`: đổi cột sang kiểu **phân loại** của pandas.
- `categorical_features="from_dtype"`: bảo mô hình: cột nào mang kiểu `category` thì xử lý như nhóm không có thứ tự.

**Cách đọc.**
- **Con số:** coi `plan` là số: **72,2%**; coi là phân loại: **73,7%**.
- **Bức tranh:** coi `plan` là số buộc cây phải cắt theo ngưỡng (gói 0 đến 57, 58 đến ...) nên khó gom đúng 100 gói rủi ro rải rác; khai báo phân loại cho phép gộp nhóm tự do. Chênh lệch ở đây **nhỏ** (1,5 điểm phần trăm, trên 6000 dòng thi): lợi ích thật nhưng khiêm tốn, vì cây sâu cũng tự xoay xở bằng nhiều lần cắt. Lợi ích tăng khi số nhóm lớn và dữ liệu nhiều. Đây cũng là lý do **không nên đưa mã số 0, 1, 2 vào mô hình như số** nếu nó chỉ là nhãn.
- **Lưu ý:** dữ liệu này dựng sao cho điều đó xảy ra; trên dữ liệu thật mức chênh phụ thuộc vào có nhóm giá trị rủi ro rải rác hay không.

**Mang về chặng 4:** khai cột nhóm là `category` để mô hình gộp nhóm giá trị thay vì cắt theo thứ tự.

## Chặng 5: Vặn núm có kiểm soát, và đừng kỳ vọng quá

**Câu hỏi:** tìm núm tốt nhất có đáng công?

Đây là lúc dùng D4: quét ngẫu nhiên vài chục tổ hợp với kiểm định chéo **trên tập học**, rồi so cài đặt tốt nhất với mặc định trên tập thi **một lần**.

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
space = {"learning_rate": loguniform(0.02, 0.3), "max_leaf_nodes": randint(8, 64), "min_samples_leaf": randint(10, 100), "l2_regularization": loguniform(1e-3, 10)}
search = RandomizedSearchCV(HistGradientBoostingClassifier(max_iter=500, early_stopping=True, random_state=0), space, n_iter=20, cv=3, n_jobs=-1, random_state=0).fit(X_train, y_train)
default = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)
print("default test:", round(default.score(X_test, y_test), 4), "| tuned test:", round(search.score(X_test, y_test), 4))
print("best:", {k: (round(float(v), 4) if not isinstance(v, (int, np.integer)) else int(v)) for k, v in search.best_params_.items()})

default test: 0.9316 | tuned test: 0.9408
best: {'l2_regularization': 4.0775, 'learning_rate': 0.0676, 'max_leaf_nodes': 39, 'min_samples_leaf': 11}


**Giải thích từng dòng**

- `loguniform(0.02, 0.3)`: bốc đều theo bậc 10 (D4).
- `randint(8, 64)`: bốc số nguyên từ 8 đến 63.
- `RandomizedSearchCV(..., n_iter=20, cv=3)`: thử 20 tổ hợp, mỗi cái chấm bằng 3 phần.
- `search.score(X_test, y_test)`: chấm **một lần** trên tập thi (niêm phong cho đến lúc này).

**Cách đọc.**
- **Con số:** mặc định **93,16%**; sau khi vặn núm **94,08%**. Tập thi 5000 dòng, nên 1 điểm phần trăm ≈ 50 dòng.
- **Bức tranh:** như D4 đã cảnh báo, lợi ích của vặn núm thường **nhỏ** khi mặc định đã tốt. Nếu chênh lệch nằm trong vài chục dòng, **xem như ngang nhau**. Thời gian vặn núm nên dành cho **cột tốt hơn** và **nhãn sạch hơn**.
- **Quy trình thực tế:** mặc định với dừng sớm, rồi nếu cần mới thử vài giá trị `learning_rate`, `max_leaf_nodes`, `l2_regularization`.

**Mang về chặng 5:** mặc định cộng dừng sớm đã rất tốt; vặn núm là bước cuối, lợi ích nhỏ.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Bản histogram nhanh hơn rất nhiều và không kém điểm ở mặc định. |
| 1 | `max_iter` lớn + dừng sớm + `learning_rate` nhỏ vừa phải. |
| 2 | Phanh an toàn: ít lá, lá nhiều dòng, phạt độ lớn; quan trọng khi dữ liệu ít. |
| 3 | Nhận `NaN` trực tiếp; nhưng vẫn phải hiểu vì sao ô trống xuất hiện. |
| 4 | Khai cột nhóm là `category` thay vì mã số có vẻ có thứ tự. |
| 5 | Mặc định đã tốt; vặn núm là bước cuối với lợi ích nhỏ. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Gom giá trị thành khoảng rồi mới tìm chỗ cắt | biểu đồ cột (histogram) hóa | `HistGradientBoostingClassifier` |
| Số lần sửa tối đa | số vòng tối đa | `max_iter` |
| Phần mỗi lần sửa được cộng | tốc độ học | `learning_rate` |
| Tự dừng khi hết cải thiện | dừng sớm | `early_stopping`, `n_iter_no_change` |
| Số vòng thực sự đã chạy | số vòng hiệu dụng | `n_iter_` |
| Cây to hay nhỏ | số lá tối đa | `max_leaf_nodes` |
| Phạt giá trị lá lớn | chính quy hóa L2 | `l2_regularization` |
| Cột nhóm không có thứ tự | đặc trưng phân loại | `categorical_features='from_dtype'` |

**Thử thách tự làm**

1. Ở chặng 0, tăng số dòng lên 100 000. Khoảng cách tốc độ giữa hai bản thay đổi thế nào?
2. Ở chặng 3, thử tăng tỉ lệ ô trống lên 50%. Mô hình nào chịu tốt hơn?
3. Ở chặng 4, thêm `OneHotEncoder` cho `plan` (B4) rồi dùng hồi quy logistic. Kết quả so với bản phân loại ra sao?
4. Dùng `HistGradientBoostingRegressor` cho bộ `load_diabetes`. MAE so với Ridge ở D7 thế nào?

**Checklist năng lực** (tự tick khi làm được)

- [ ] Giải thích vì sao bản histogram nhanh hơn.
- [ ] Đặt `max_iter` lớn và dùng dừng sớm.
- [ ] Cho mô hình ăn thẳng bảng có ô trống và hiểu hạn chế.
- [ ] Khai cột nhóm là `category` đúng cách.
- [ ] Biết khi nào thôi vặn núm.

## Phần thưởng (Tip pro): XGBoost, LightGBM, CatBoost


Ngoài sklearn còn ba thư viện boosting nổi tiếng: **XGBoost**, **LightGBM**, **CatBoost**. Về ý tưởng, chúng là cùng họ với bài hôm nay (LightGBM đặc biệt gần với `HistGradientBoosting`). Chúng thêm nhiều tính năng, chạy trên GPU, và rất phổ biến trong cuộc thi dữ liệu bảng. Học viên đã nắm E3 và E4 có thể đọc tài liệu của chúng mà không bỡ ngỡ: các núm `learning_rate`, số vòng, số lá, phạt L2 đều quen thuộc. Với khóa này ta giữ trong sklearn để chạy ngay trên Colab mà không cần cài thêm.